<a href="https://colab.research.google.com/github/prof-atritiack/2TDSPO-CP2-GEN_AI/blob/main/CP2_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHECKPOINT 02: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Moisés Barsoti Andrade de Oliveira  RM565049

**Tema escolhido:** Assistente de Casa Inteligente

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória e troca de modelo
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
!pip install huggingface_hub google-genai gradio -q

In [2]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [10]:
MODELO_HF     = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf     = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [11]:
SYSTEM_PROMPT = """
Você é um assistente virtual especializado em casas inteligentes e automação residencial.

Seu objetivo é auxiliar os usuários na utilização, configuração e solução de problemas relacionados a dispositivos inteligentes.

Você pode responder perguntas sobre:
- Automação residencial e Internet das Coisas (IoT).
- Sensores de presença, temperatura, luminosidade e movimento.
- Dispositivos como lâmpadas inteligentes, câmeras e fechaduras eletrônicas.
- Integração com assistentes virtuais como Alexa e Google Home.
- Conectividade Wi-Fi, Bluetooth e protocolos de comunicação IoT.
- Economia de energia e segurança residencial.

Regras de comportamento:
1. Responda sempre em português brasileiro.
2. Utilize linguagem clara, objetiva e acessível para iniciantes.
3. Limite suas respostas a no máximo 5 frases.
4. Quando necessário, forneça orientações em etapas simples.
5. Não invente informações técnicas ou funcionalidades de dispositivos.
6. Não forneça instruções que comprometam a segurança dos usuários.
7. Caso a pergunta não tenha relação com casas inteligentes ou IoT residencial, informe educadamente que o assunto está fora da sua especialidade.
"""


---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [12]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    mensagens = [
        {"role": "system", "content": system},
        {"role": "user",   "content": pergunta},
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [13]:

# >>> PERSONALIZE: versões do system prompt.

versoes_system = {
    "Sem restrição": """
    Você é um assistente virtual prestativo.
    Responda às perguntas do usuário em português brasileiro,
    utilizando uma linguagem clara e objetiva.
    Você pode responder perguntas sobre qualquer assunto.
    """,

    "Com restrição de tema": """
    Você é um assistente virtual especializado em casas inteligentes
    e automação residencial.

    Responda exclusivamente perguntas relacionadas a:
    - Internet das Coisas (IoT) aplicada a residências.
    - Sensores de presença, movimento, temperatura e luminosidade.
    - Lâmpadas inteligentes, câmeras e fechaduras eletrônicas.
    - Assistentes virtuais como Alexa e Google Home.
    - Conectividade e segurança de dispositivos inteligentes.

    Caso o usuário faça uma pergunta fora da sua especialidade,
    informe educadamente que você só pode ajudar com assuntos
    relacionados a casas inteligentes e automação residencial.

    Responda sempre em português brasileiro.
    """,

    "Com restrição de tema e formato": """
    Você é um assistente virtual especializado em casas inteligentes
    e automação residencial.

    Responda exclusivamente perguntas sobre automação residencial,
    sensores, dispositivos inteligentes e IoT residencial.

    Regras obrigatórias:
    1. Responda sempre em português brasileiro.
    2. Utilize linguagem simples, clara e objetiva.
    3. Limite suas respostas a no máximo 3 frases.
    4. Não utilize listas, tópicos ou marcadores.
    5. Não invente informações técnicas.
    6. Se a pergunta estiver fora do tema, informe educadamente
       que você só responde dúvidas sobre casas inteligentes.
    """
}

# >>> PERSONALIZE: perguntas para testar os guardrails.

pergunta_dentro = "Como um sensor de presença pode ajudar a economizar energia em uma casa inteligente?"

pergunta_fora = "Quais são as principais causas da Revolução Francesa?"

# Execução dos testes.

for nome, system in versoes_system.items():
    print(f"\n{'='*55}")
    print(f"  {nome}")
    print(f"{'='*55}")

    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_gemini(pergunta_dentro, system))

    print(f"\n[Fora do escopo] {pergunta_fora}")
    print(perguntar_gemini(pergunta_fora, system))


  Sem restrição

[Dentro do escopo] Como um sensor de presença pode ajudar a economizar energia em uma casa inteligente?
Um sensor de presença é um dos dispositivos mais eficazes em uma casa inteligente para reduzir o consumo de energia. Ele funciona automatizando o controle de luzes, climatização e eletrodomésticos com base na ocupação real dos ambientes. 

Aqui estão as principais formas de como ele ajuda a economizar energia:

1. **Fim das luzes acesas por esporte:** Quantas vezes você já esqueceu a luz do banheiro, do corredor ou da garagem acesa ao sair? O sensor garante que as luzes só acionem quando houver alguém no cômodo e, o mais importante, **apagam automaticamente** após um período sem detectar movimento.

2. **Otimização do ar-condicionado e aquecimento:** Integrado a um sistema inteligente, o sensor pode avisar quando um quarto está vazio e desligar (ou reduzir a potência) do ar-condicionado ou aquecedor, evitando que você climatize cômodos que ninguém está usando.

3. *

**Observações do grupo (Etapa 1):**

**1. O assistente sem restrição respondeu à pergunta fora do escopo?**

Sim. O assistente respondeu à pergunta sobre as principais causas da Revolução Francesa, mesmo sendo um assunto sem relação com casas inteligentes. Isso ocorreu porque o system prompt não possuía restrições de tema.

**2. Com o guardrail de tema, ele recusou? A recusa foi educada?**

Sim. Ao aplicar a restrição de tema, o assistente recusou educadamente a pergunta sobre a Revolução Francesa e explicou que sua especialidade envolve automação residencial, sensores e dispositivos IoT. Isso demonstrou que os guardrails ajudam a limitar os assuntos abordados pela inteligência artificial.

**3. A restrição de formato mudou o estilo da resposta?**

Sim. Com a restrição de tema e formato, o assistente apresentou respostas mais curtas, diretas e organizadas em no máximo três frases, sem utilizar listas ou tópicos. Assim, percebemos que as instruções do system prompt influenciam tanto o conteúdo quanto a estrutura das respostas.

**Conclusão:** Os testes realizados com o modelo Gemini demonstraram que os guardrails podem controlar o comportamento do assistente, restringindo assuntos e padronizando o formato das respostas.

---
## Etapa 2: Comparação com temperatura

Na atividade anterior o grupo comparou Hugging Face e Gemini com o mesmo prompt.
Aqui o grupo compara o efeito da **temperatura** nos dois modelos.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

A mesma pergunta é enviada com cada valor de temperatura para os dois modelos.

In [8]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [15]:

import time

perguntas = [
    "Quais sensores são mais usados em uma casa inteligente?",
    "Quais cuidados de segurança devo ter com câmeras conectadas?",
]

for pergunta in perguntas:
    print(f"\nPERGUNTA: {pergunta}")

    for temp in [0.0, 1.0]:
        print(f"\n--- temperature={temp} ---")

        for tentativa in range(3):
            try:
                resposta = perguntar_gemini(
                    pergunta,
                    SYSTEM_PROMPT,
                    temperatura=temp
                )

                print(f"[Gemini] {resposta}")
                break

            except Exception as erro:
                if "503" in str(erro) and tentativa < 2:
                    print("Modelo ocupado. Aguardando...")
                    time.sleep(10 * (tentativa + 1))
                else:
                    print(f"[Gemini] Erro: {erro}")
                    break

        time.sleep(5)

    print("\n" + "=" * 55)



PERGUNTA: Quais sensores são mais usados em uma casa inteligente?

--- temperature=0.0 ---
[Gemini] Os sensores mais populares em uma casa inteligente são os de presença e movimento, ideais para acender luzes automaticamente e aumentar a segurança. 
Os sensores de abertura de portas e janelas também são muito usados para monitoramento e controle de ar-condicionado. 
Outros dispositivos comuns são os sensores de temperatura e umidade, que ajudam a regular o clima dos ambientes. 
Também destacam-se os sensores de luminosidade, que ajustam a intensidade das lâmpadas conforme a luz natural. 
Por fim, os sensores de vazamento de água e fumaça são essenciais para prevenir acidentes graves na residência.

--- temperature=1.0 ---
[Gemini] Os sensores mais populares em uma casa inteligente são os de presença e movimento, ideais para acender luzes automaticamente e aumentar a segurança. 
Os sensores de abertura de portas e janelas também são muito usados para monitoramento e controle do ar-cond

**Observações do grupo (Etapa 2):**

**1. Com `temperature=0.0`, as respostas dos dois modelos ficaram parecidas entre si?**

Não foi possível comparar os dois modelos, pois o Hugging Face apresentou erro 402 devido à falta de créditos. Entretanto, conseguimos executar os testes com o Gemini utilizando temperatura 0.0, obtendo respostas claras, objetivas e relacionadas ao tema de casas inteligentes.

**2. Com `temperature=1.0`, as respostas ficaram mais variadas?**

Parcialmente. Na pergunta sobre sensores inteligentes, as respostas foram praticamente idênticas nas duas temperaturas. Já na pergunta sobre segurança de câmeras, a temperatura 1.0 apresentou informações diferentes, incluindo protocolos WPA2/WPA3, VPN e UPnP. Isso demonstrou que temperaturas maiores podem gerar variações, mas não necessariamente em todas as respostas.

**3. Qual combinação de modelo e temperatura vocês consideram melhor para o tema?**

Consideramos o Gemini com temperatura 0.0 uma escolha adequada para nosso assistente de casas inteligentes, pois apresentou respostas objetivas e consistentes. Como o assistente fornece orientações técnicas sobre automação residencial e segurança, priorizamos a previsibilidade das respostas. Entretanto, não foi possível determinar qual dos dois modelos seria superior, devido à indisponibilidade do Hugging Face.

**Conclusão:** Os testes demonstraram a influência da temperatura na geração de respostas e a importância de selecionar configurações adequadas ao objetivo do assistente. A comparação completa entre os modelos permanece pendente.

---
## Etapa 3: Chat com memória e troca de modelo

Na atividade anterior o chat usava apenas o Hugging Face.
Aqui o grupo pode trocar o modelo durante a conversa, mantendo o histórico.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas
- `modelo hf` — troca para Hugging Face
- `modelo gemini` — troca para Gemini

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", troque o modelo e pergunte de novo.

In [16]:

def responder_chat(historico, modelo):
    if modelo == "hf":
        resposta = cliente_hf.chat_completion(
            messages=historico,
            max_tokens=300
        )
        return resposta.choices[0].message.content

    # Gemini: converte o histórico para o formato do modelo.
    conteudos = []

    for msg in historico:
        if msg["role"] == "system":
            continue

        papel = "model" if msg["role"] == "assistant" else "user"

        conteudos.append({
            "role": papel,
            "parts": [{"text": msg["content"]}]
        })

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=500,
        ),
    )

    if not resposta.text:
        raise ValueError("O Gemini não retornou uma resposta.")

    return resposta.text


In [17]:

historico = [{"role": "system", "content": SYSTEM_PROMPT}]
modelo_atual = "gemini"

print(f"Chat iniciado com modelo: {modelo_atual}")
print("Comandos: sair | limpar | historico | modelo hf | modelo gemini\n")

while True:
    entrada = input("Você: ").strip()

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens: {len(historico)} | modelo: {modelo_atual})\n")
        continue

    if entrada.lower() in ("modelo hf", "modelo gemini"):
        modelo_atual = entrada.split()[1]
        print(f"\n(modelo trocado para: {modelo_atual})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    try:
        texto = responder_chat(historico, modelo_atual)

        historico.append({
            "role": "assistant",
            "content": texto
        })

        print(f"\n[{modelo_atual}] {texto}\n")

    except Exception as erro:
        historico.pop()

        print(f"\nErro ao consultar {modelo_atual}: {erro}")
        print("A mensagem não foi salva no histórico.")
        print("Você pode trocar de modelo e tentar novamente.\n")


Chat iniciado com modelo: gemini
Comandos: sair | limpar | historico | modelo hf | modelo gemini

Você: Meu nome é Moisés e utilizo sensores inteligentes.

[gemini] Olá, Moisés! É um prazer ajudar você a otimizar o uso dos seus sensores inteligentes na automação da sua casa. Como posso auxiliar hoje com a configuração, rotinas ou solução de algum problema com seus dispositivos?

Você: Qual é meu nome e quais dispositivos utilizo?

[gemini] O seu nome é Moisés e você mencionou que utiliza sensores inteligentes na sua casa. Como sou um assistente virtual, não tenho acesso aos dados privados da sua residência para saber exatamente quais modelos ou marcas de sensores você possui. Se quiser me contar quais dispositivos usa, poderei ajudar melhor na configuração e nas rotinas deles!

Você: historico

(mensagens: 5 | modelo: gemini)

Você: modelo hf

(modelo trocado para: hf)

Você: Qual é meu nome?

Erro ao consultar hf: Client error '402 Payment Required' for url 'https://router.huggingface

In [18]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system]  Você é um assistente virtual especializado em casas inteligentes e automação re
[user] Meu nome é Moisés e utilizo sensores inteligentes.
[assistant] Olá, Moisés! É um prazer ajudar você a otimizar o uso dos seus sensores intelige
[user] Qual é meu nome e quais dispositivos utilizo?
[assistant] O seu nome é Moisés e você mencionou que utiliza sensores inteligentes na sua ca


**Observações do grupo (Etapa 3):**

**1. Ao trocar de modelo no meio da conversa, o novo modelo lembrou do que foi dito?**

Durante os testes, o Gemini conseguiu recuperar corretamente o nome do usuário e os dispositivos mencionados anteriormente. Ao trocar para o Hugging Face, o sistema alterou o modelo selecionado, porém não foi possível verificar se ele utilizaria o histórico, pois a API retornou erro 402 devido à falta de créditos.

**2. Por que isso acontece (ou não acontece)?**

O histórico da conversa é armazenado em uma lista Python chamada `historico`, contendo as mensagens do usuário e as respostas do assistente. A cada nova pergunta, essas informações são enviadas ao modelo selecionado, permitindo que ele utilize o contexto anterior. Como a aplicação mantém o histórico independentemente do modelo, uma tentativa de troca não apaga as mensagens anteriores.

**3. O que muda na forma como o histórico é enviado para o Gemini em relação ao Hugging Face?**

No Hugging Face, o histórico é enviado diretamente como uma lista de mensagens com os papéis `system`, `user` e `assistant`. No Gemini, é necessário converter os papéis das mensagens, substituindo `assistant` por `model` e organizando o conteúdo em `parts`. Além disso, o system prompt é enviado separadamente pelo parâmetro `system_instruction`.

**Conclusão:** O teste demonstrou que a aplicação consegue armazenar e reutilizar mensagens anteriores com Gemini, preservando o histórico mesmo quando ocorre um erro na troca de modelo. A comunicação efetiva entre os dois modelos permanece pendente devido à indisponibilidade do Hugging Face.

---
## Etapa 4: Interface web com Gradio e controle de temperatura

Na atividade anterior a interface tinha o seletor de modelo.
Aqui o grupo adiciona um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [19]:

import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo

    if isinstance(conteudo, list):
        return " ".join(
            item.get("text", "")
            for item in conteudo
            if isinstance(item, dict)
        )

    return str(conteudo)


def responder(mensagem, historico_gradio, provedor, temperatura):
    try:
        if provedor == "Hugging Face":
            mensagens = [
                {"role": "system", "content": SYSTEM_PROMPT}
            ]

            for msg in historico_gradio:
                mensagens.append({
                    "role": msg["role"],
                    "content": texto_da_mensagem(msg["content"])
                })

            mensagens.append({
                "role": "user",
                "content": mensagem
            })

            resposta = cliente_hf.chat_completion(
                messages=mensagens,
                max_tokens=500,
                temperature=temperatura,
            )

            return resposta.choices[0].message.content

        else:
            conteudos = []

            for msg in historico_gradio:
                papel = (
                    "model"
                    if msg["role"] == "assistant"
                    else "user"
                )

                conteudos.append({
                    "role": papel,
                    "parts": [{
                        "text": texto_da_mensagem(msg["content"])
                    }]
                })

            conteudos.append({
                "role": "user",
                "parts": [{"text": mensagem}]
            })

            resposta = cliente_gemini.models.generate_content(
                model=MODELO_GEMINI,
                contents=conteudos,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    max_output_tokens=500,
                    temperature=temperatura,
                ),
            )

            return resposta.text or "O modelo não retornou uma resposta."

    except Exception as erro:
        if "402" in str(erro):
            return (
                "O Hugging Face está sem créditos disponíveis. "
                "Selecione Gemini para continuar."
            )

        if "503" in str(erro):
            return (
                "O modelo está temporariamente sobrecarregado. "
                "Aguarde alguns instantes e tente novamente."
            )

        return f"Erro ao consultar o modelo: {erro}"


In [20]:

gr.ChatInterface(
    fn=responder,

    additional_inputs=[
        gr.Radio(
            ["Hugging Face", "Gemini"],
            value="Gemini",
            label="Modelo de Inteligência Artificial"
        ),

        gr.Slider(
            minimum=0.0,
            maximum=1.0,
            step=0.1,
            value=0.0,
            label="Temperatura"
        ),
    ],

    title="SmartHome AI - Assistente de Casa Inteligente",

    description=(
        "Assistente virtual especializado em automação residencial "
        "e Internet das Coisas (IoT). Tire dúvidas sobre sensores, "
        "dispositivos conectados, segurança e economia de energia."
    ),

    examples=[
        ["Como funciona uma lâmpada inteligente?", "Gemini", 0.0],
        ["Como economizar energia com sensores?", "Gemini", 0.3],
        ["Como proteger câmeras inteligentes?", "Gemini", 0.7],
        ["Quais dispositivos posso integrar à Alexa?", "Gemini", 1.0],
    ],

).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a201dea2756ac756ca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**2. O que mudou entre as temperaturas 0.0 e 1.0?**

Nos testes realizados anteriormente com o Gemini, observamos que as respostas sobre sensores inteligentes foram praticamente idênticas nas duas temperaturas. Já nas perguntas sobre segurança de câmeras, a temperatura 1.0 apresentou recomendações diferentes, incluindo WPA2/WPA3, VPN e UPnP. Isso demonstra que temperaturas maiores podem gerar respostas mais variadas, embora nem sempre exista uma diferença significativa.

**3. Qual combinação funcionou melhor?**

Consideramos o Gemini com temperatura 0.0 a opção mais adequada para o nosso assistente de casas inteligentes, pois apresentou respostas claras, objetivas e consistentes. O Hugging Face não pôde ser avaliado completamente devido à indisponibilidade de créditos na API.

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [21]:

%%writefile app_avaliacao.py
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google import genai
from google.genai import types

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

MODELO_GEMINI = os.environ.get(
    "MODELO_GEMINI",
    "gemini-3.5-flash-lite"
)

SYSTEM_PROMPT = os.environ.get(
    "SYSTEM_PROMPT",
    "Você é um assistente de casa inteligente."
)

app = FastAPI(
    title="SmartHome AI",
    description="API de casa inteligente com sessões independentes"
)

# Histórico por sessão
sessoes: dict[str, list] = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    historico = sessoes.get(pergunta.session_id, [])

    conteudos = historico + [
        {
            "role": "user",
            "parts": [{"text": pergunta.mensagem}]
        }
    ]

    try:
        resposta = client.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.0,
                max_output_tokens=500
            )
        )

        if not resposta.text:
            raise ValueError("Resposta vazia.")

    except Exception as erro:
        raise HTTPException(
            status_code=503,
            detail=f"Falha na geração: {erro}"
        )

    # Salva o histórico somente após uma resposta válida
    sessoes[pergunta.session_id] = conteudos + [
        {
            "role": "model",
            "parts": [{"text": resposta.text}]
        }
    ]

    return {
        "session_id": pergunta.session_id,
        "resposta": resposta.text
    }

@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)

    return {
        "status": "sessão apagada",
        "session_id": session_id
    }


Writing app_avaliacao.py


In [22]:

!pip install fastapi uvicorn requests -q

import os
import subprocess
import time

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT
os.environ["MODELO_GEMINI"] = MODELO_GEMINI

servidor = subprocess.Popen([
    "uvicorn",
    "app_avaliacao:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

time.sleep(5)

print("Servidor iniciado em http://localhost:8000")


Servidor iniciado em http://localhost:8000


In [23]:

import requests

URL = "http://localhost:8000"

# Limpar sessões de possíveis testes anteriores
for sessao in ["sessao_a", "sessao_b"]:
    requests.delete(f"{URL}/sessao/{sessao}", timeout=15)

# Sessão A — informa o nome
r_inicio = requests.post(
    f"{URL}/chat",
    json={
        "session_id": "sessao_a",
        "mensagem": "Meu nome é Ana."
    },
    timeout=90
)
r_inicio.raise_for_status()

# Sessão A — pergunta o nome
r_a = requests.post(
    f"{URL}/chat",
    json={
        "session_id": "sessao_a",
        "mensagem": "Qual é meu nome?"
    },
    timeout=90
)
r_a.raise_for_status()

# Sessão B — não informou o nome
r_b = requests.post(
    f"{URL}/chat",
    json={
        "session_id": "sessao_b",
        "mensagem": "Qual é meu nome?"
    },
    timeout=90
)
r_b.raise_for_status()

print("Sessão A:", r_a.json()["resposta"])
print("Sessão B:", r_b.json()["resposta"])


Sessão A: Seu nome é Ana, conforme você me disse anteriormente. Como posso ajudar você hoje com a sua casa inteligente?
Sessão B: Como sou um assistente virtual especializado em casas inteligentes, eu não tenho acesso às suas informações pessoais e não sei o seu nome. Posso ajudar você a configurar dispositivos, criar automações ou tirar dúvidas sobre tecnologia residencial. Como posso auxiliar na sua casa inteligente hoje?


In [24]:
servidor.terminate()
servidor.wait(timeout=10)

print("Servidor encerrado.")

Servidor encerrado.


**Observações do grupo (Etapa 5 — Bônus):**

**1. A sessão A respondeu o nome corretamente? E a sessão B?**

Sim. A sessão A identificou corretamente o nome Ana, que havia sido informado anteriormente. Já a sessão B respondeu que não conhecia o nome do usuário, demonstrando que as informações de uma sessão não foram compartilhadas com a outra.

**2. O que mudou nessa API em relação à atividade anterior?**

Na atividade anterior, a API não armazenava o histórico das conversas entre as requisições. Nesta versão, implementamos um dicionário Python chamado `sessoes`, que armazena os históricos separadamente utilizando o `session_id`. Dessa forma, cada sessão consegue manter seu próprio contexto durante a execução do servidor.

**3. Cite um caso de uso real onde sessões independentes seriam necessárias.**

Um exemplo seria um aplicativo de automação residencial utilizado por diferentes famílias. Cada família poderia conversar com o assistente sobre seus sensores, câmeras e dispositivos inteligentes sem que as informações de uma residência fossem misturadas com as de outra.

**Conclusão:** A implementação com FastAPI e Gemini demonstrou o funcionamento de sessões independentes, permitindo preservar o contexto de cada conversa sem compartilhar informações entre sessões diferentes.